In [1]:
!pip install -q joblib

import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.multioutput import MultiOutputClassifier

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

RANDOM_STATE = 42

In [2]:
uploaded = files.upload()

file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)

print("Shape:", df.shape)
df.head()

Saving adult.csv to adult.csv
Shape: (8000, 14)


,age_years,sex,heart_rate_bpm,respiratory_rate_bpm,systolic_bp_mmHg,spo2_percent,temperature_c,level_of_consciousness,chief_complaint_category,duration_days,comorbidity_count,pain_distress_score_0_10,clinical_disposition,severity_score
0,57,Female,120.4,23.1,117.6,94.0,36.8,Alert,Respiratory,0,1,1,Treat + monitor,Medium
1,70,Female,127.7,23.2,120.5,96.1,38.7,Alert,Fever/Infection,2,1,9,Treat + monitor,Medium
2,55,Female,90.0,17.2,131.9,97.3,37.6,Alert,Urinary,3,1,0,Treat locally,Low
3,64,Female,84.4,20.9,107.8,93.8,38.3,Alert,Chest pain,3,1,10,Treat + monitor,Medium
4,52,Male,100.3,30.7,99.2,89.7,37.5,Alert,Respiratory,3,1,6,Stabilize + refer,High


In [3]:
expected_cols = [
    'age_years','sex','heart_rate_bpm','respiratory_rate_bpm',
    'systolic_bp_mmHg','spo2_percent','temperature_c',
    'level_of_consciousness','chief_complaint_category',
    'duration_days','comorbidity_count','pain_distress_score_0_10',
    'clinical_disposition','severity_score'
]

df = df[expected_cols].drop_duplicates().reset_index(drop=True)

# Strip spaces
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

df.head()

,age_years,sex,heart_rate_bpm,respiratory_rate_bpm,systolic_bp_mmHg,spo2_percent,temperature_c,level_of_consciousness,chief_complaint_category,duration_days,comorbidity_count,pain_distress_score_0_10,clinical_disposition,severity_score
0,57,Female,120.4,23.1,117.6,94.0,36.8,Alert,Respiratory,0,1,1,Treat + monitor,Medium
1,70,Female,127.7,23.2,120.5,96.1,38.7,Alert,Fever/Infection,2,1,9,Treat + monitor,Medium
2,55,Female,90.0,17.2,131.9,97.3,37.6,Alert,Urinary,3,1,0,Treat locally,Low
3,64,Female,84.4,20.9,107.8,93.8,38.3,Alert,Chest pain,3,1,10,Treat + monitor,Medium
4,52,Male,100.3,30.7,99.2,89.7,37.5,Alert,Respiratory,3,1,6,Stabilize + refer,High


In [4]:
clinical_order = [
    'Treat locally',
    'Treat + monitor',
    'Stabilize + refer',
    'Emergency referral'
]

severity_order = ['Low','Medium','High']

clinical_map = {v:i for i,v in enumerate(clinical_order)}
severity_map = {v:i for i,v in enumerate(severity_order)}

inv_clinical = {i:v for v,i in clinical_map.items()}
inv_severity = {i:v for v,i in severity_map.items()}

X = df.drop(['clinical_disposition','severity_score'], axis=1)

y = pd.DataFrame({
    'clinical_disposition': df['clinical_disposition'].map(clinical_map),
    'severity_score': df['severity_score'].map(severity_map)
})

X.head(), y.head()

(   age_years     sex  heart_rate_bpm  respiratory_rate_bpm  systolic_bp_mmHg  \
 0         57  Female           120.4                  23.1             117.6   
 1         70  Female           127.7                  23.2             120.5   
 2         55  Female            90.0                  17.2             131.9   
 3         64  Female            84.4                  20.9             107.8   
 4         52    Male           100.3                  30.7              99.2   
 
    spo2_percent  temperature_c level_of_consciousness  \
 0          94.0           36.8                  Alert   
 1          96.1           38.7                  Alert   
 2          97.3           37.6                  Alert   
 3          93.8           38.3                  Alert   
 4          89.7           37.5                  Alert   
 
   chief_complaint_category  duration_days  comorbidity_count  \
 0              Respiratory              0                  1   
 1          Fever/Infection     

In [5]:
stratify_key = df['clinical_disposition'] + "_" + df['severity_score']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=stratify_key
)

print(X_train.shape, X_test.shape)

(6400, 12) (1600, 12)


In [6]:
num_cols = [
    'age_years','heart_rate_bpm','respiratory_rate_bpm',
    'systolic_bp_mmHg','spo2_percent','temperature_c',
    'duration_days','comorbidity_count','pain_distress_score_0_10'
]

cat_cols = [
    'sex','level_of_consciousness','chief_complaint_category'
]

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

In [7]:
models = {
    "logistic": LogisticRegression(max_iter=3000),
    "rf": RandomForestClassifier(n_estimators=300, random_state=42),
    "extra": ExtraTreesClassifier(n_estimators=400, random_state=42)
}

pipelines = {}

for name, model in models.items():
    pipelines[name] = Pipeline([
        ('prep', preprocessor),
        ('model', MultiOutputClassifier(model))
    ])

In [8]:
trained = {}

for name, pipe in pipelines.items():
    print("Training:", name)
    pipe.fit(X_train, y_train)
    trained[name] = pipe

Training: logistic
Training: rf
Training: extra


In [9]:
def evaluate(model, name):
    pred = model.predict(X_test)

    y_true_disp = y_test['clinical_disposition']
    y_true_sev  = y_test['severity_score']

    y_pred_disp = pred[:,0]
    y_pred_sev  = pred[:,1]

    print("\n====", name, "====")

    print("Disposition F1:", f1_score(y_true_disp, y_pred_disp, average='macro'))
    print("Severity F1   :", f1_score(y_true_sev, y_pred_sev, average='macro'))

    print("\nDisposition Report")
    print(classification_report(y_true_disp, y_pred_disp, target_names=clinical_order))

    print("\nSeverity Report")
    print(classification_report(y_true_sev, y_pred_sev, target_names=severity_order))

In [10]:
scores = []

for name, model in trained.items():
    evaluate(model, name)

    pred = model.predict(X_test)
    f1_disp = f1_score(y_test['clinical_disposition'], pred[:,0], average='macro')
    f1_sev  = f1_score(y_test['severity_score'], pred[:,1], average='macro')

    score = 0.65*f1_disp + 0.35*f1_sev
    scores.append((name, score))

scores = sorted(scores, key=lambda x: x[1], reverse=True)
scores


==== logistic ====
Disposition F1: 0.7595836995716896
Severity F1   : 0.8326256319158749

Disposition Report
                    precision    recall  f1-score   support

     Treat locally       0.82      0.85      0.84       400
   Treat + monitor       0.68      0.67      0.68       400
 Stabilize + refer       0.66      0.68      0.67       400
Emergency referral       0.87      0.83      0.85       400

          accuracy                           0.76      1600
         macro avg       0.76      0.76      0.76      1600
      weighted avg       0.76      0.76      0.76      1600


Severity Report
              precision    recall  f1-score   support

         Low       0.86      0.88      0.87       451
      Medium       0.74      0.73      0.73       485
        High       0.89      0.89      0.89       664

    accuracy                           0.84      1600
   macro avg       0.83      0.83      0.83      1600
weighted avg       0.84      0.84      0.84      1600


==== rf 

[('rf', 0.9393140027958664),
 ('extra', 0.8920924125899192),
 ('logistic', 0.7851483758921545)]

In [11]:
best_name = scores[0][0]
best_model = trained[best_name]

print("Best model:", best_name)

Best model: rf


In [12]:
bundle = {
    "model": best_model,
    "features": X.columns.tolist(),
    "inv_clinical": inv_clinical,
    "inv_severity": inv_severity
}

joblib.dump(bundle, "adult_model.joblib")

files.download("adult_model.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
sample = {
    'age_years': 60,
    'sex': 'Male',
    'heart_rate_bpm': 120,
    'respiratory_rate_bpm': 28,
    'systolic_bp_mmHg': 95,
    'spo2_percent': 90,
    'temperature_c': 38.5,
    'level_of_consciousness': 'Alert',
    'chief_complaint_category': 'Respiratory',
    'duration_days': 2,
    'comorbidity_count': 2,
    'pain_distress_score_0_10': 7
}

sample_df = pd.DataFrame([sample])

pred = best_model.predict(sample_df)[0]

print("Clinical:", inv_clinical[int(pred[0])])
print("Severity:", inv_severity[int(pred[1])])

Clinical: Stabilize + refer
Severity: High
